In [ ]:
import pandas as pd
import numpy as np

# 1. Data Loading & Basic Configuration
df = pd.read_excel(
    "E:\\...\\03Loss_Mainland_Combined_20251119.xlsx",
    sheet_name="SSP585"
)[lambda x: x['ID'].notna()]

# Configuration Constants
TARGET_COLS = ['Total economic loss 2050', 'Economic loss 2050', 'Health loss 2050']
TARGET_CONTINENTS = ['Asia', 'Africa', 'Europe']
QUANTILE = 0.1  # 10th percentile
HIGH_QUANTILE = 0.9  # 90th percentile (for Ratio ≥ threshold)

# 2. Core Calculation Functions
def calc_quantile_stats(data, group_col=None):
    if group_col is None:
        # Global (World) calculation
        q_val = data['MeanRatio'].quantile(QUANTILE)
        filtered = data[data['MeanRatio'] <= q_val]
        mean_dict = {col: filtered[col].mean().round(6) for col in TARGET_COLS}
        count_dict = {col: len(filtered) for col in TARGET_COLS}
        return mean_dict, count_dict, q_val
    else:
        # Calculation by group
        mean_res, count_res, q_res = {}, {}, {}
        for group in data[group_col].unique():
            g_data = data[data[group_col] == group]
            if len(g_data) == 0:
                continue
            q_val = g_data['MeanRatio'].quantile(QUANTILE)
            filtered = g_data[g_data['MeanRatio'] <= q_val]
            mean_res[group] = {col: filtered[col].mean().round(6) for col in TARGET_COLS}
            count_res[group] = {col: len(filtered) for col in TARGET_COLS}
            q_res[group] = q_val
        return mean_res, count_res, q_res

def calc_high_quantile_stats(data, group_col=None):
    if group_col is None:
        # Global (World) calculation
        q_val = data['MeanRatio'].quantile(HIGH_QUANTILE)
        filtered = data[data['MeanRatio'] >= q_val]
        mean_dict = {col: filtered[col].mean().round(6) for col in TARGET_COLS}
        count_dict = {col: len(filtered) for col in TARGET_COLS}
        return mean_dict, count_dict, q_val
    else:
        # Calculation by group
        mean_res, count_res, q_res = {}, {}, {}
        for group in data[group_col].unique():
            g_data = data[data[group_col] == group]
            if len(g_data) == 0:
                continue
            q_val = g_data['MeanRatio'].quantile(HIGH_QUANTILE)
            filtered = g_data[g_data['MeanRatio'] >= q_val]
            mean_res[group] = {col: filtered[col].mean().round(6) for col in TARGET_COLS}
            count_res[group] = {col: len(filtered) for col in TARGET_COLS}
            q_res[group] = q_val
        return mean_res, count_res, q_res

def calc_all_ratio_stats(data, group_col=None):
    """Calculate mean loss and sample count for all Ratio data in specified groups"""
    if group_col is None:
        # Global (World) calculation
        mean_dict = {col: data[col].mean().round(6) for col in TARGET_COLS}
        count_dict = {col: len(data) for col in TARGET_COLS}
        return mean_dict, count_dict
    else:
        # Calculation by group
        mean_res, count_res = {}, {}
        for group in data[group_col].unique():
            g_data = data[data[group_col] == group]
            if len(g_data) == 0:
                continue
            mean_res[group] = {col: g_data[col].mean().round(6) for col in TARGET_COLS}
            count_res[group] = {col: len(g_data) for col in TARGET_COLS}
        return mean_res, count_res

# 3. Calculate Global (World) Statistics
# 10th percentile (low Ratio)
quant10_mean_world, quant10_count_world, quant10_val_world = calc_quantile_stats(df)
# 90th percentile (high Ratio)
quant90_mean_world, quant90_count_world, quant90_val_world = calc_high_quantile_stats(df)
# All Ratio data
all_mean_world, all_count_world = calc_all_ratio_stats(df)

# Compile World results
world_result = []
for col in TARGET_COLS:
    # Low Ratio (≤10th percentile)
    world_result.append({
        'Continent': 'World',
        'Loss Indicator': col,
        'Stat Type': 'Ratio ≤ 10th percentile',
        'Mean Loss': quant10_mean_world[col],
        'Sample Count': quant10_count_world[col],
        '10th Percentile': quant10_val_world,
        '90th Percentile': quant90_val_world
    })
    # High Ratio (≥90th percentile)
    world_result.append({
        'Continent': 'World',
        'Loss Indicator': col,
        'Stat Type': 'Ratio ≥ 90th percentile',
        'Mean Loss': quant90_mean_world[col],
        'Sample Count': quant90_count_world[col],
        '10th Percentile': quant10_val_world,
        '90th Percentile': quant90_val_world
    })
    # All Ratio data
    world_result.append({
        'Continent': 'World',
        'Loss Indicator': col,
        'Stat Type': 'All Ratio data',
        'Mean Loss': all_mean_world[col],
        'Sample Count': all_count_world[col],
        '10th Percentile': quant10_val_world,
        '90th Percentile': quant90_val_world
    })

# 4. Calculate Target Continent Statistics
continent_result = []
# Filter target continents
df_cont = df[df['continent'].isin(TARGET_CONTINENTS)]
# 10th percentile (low Ratio)
quant10_mean_cont, quant10_count_cont, quant10_val_cont = calc_quantile_stats(df_cont, 'continent')
# 90th percentile (high Ratio)
quant90_mean_cont, quant90_count_cont, quant90_val_cont = calc_high_quantile_stats(df_cont, 'continent')
# All Ratio data
all_mean_cont, all_count_cont = calc_all_ratio_stats(df_cont, 'continent')

# Compile continent results
for cont in TARGET_CONTINENTS:
    df_single = df[df['continent'] == cont]
    if len(df_single) == 0:
        continue
    
    for col in TARGET_COLS:
        # Low Ratio (≤10th percentile)
        continent_result.append({
            'Continent': cont,
            'Loss Indicator': col,
            'Stat Type': 'Ratio ≤ 10th percentile',
            'Mean Loss': quant10_mean_cont.get(cont, {}).get(col, np.nan),
            'Sample Count': quant10_count_cont.get(cont, {}).get(col, 0),
            '10th Percentile': quant10_val_cont.get(cont, np.nan),
            '90th Percentile': quant90_val_cont.get(cont, np.nan)
        })
        # High Ratio (≥90th percentile)
        continent_result.append({
            'Continent': cont,
            'Loss Indicator': col,
            'Stat Type': 'Ratio ≥ 90th percentile',
            'Mean Loss': quant90_mean_cont.get(cont, {}).get(col, np.nan),
            'Sample Count': quant90_count_cont.get(cont, {}).get(col, 0),
            '10th Percentile': quant10_val_cont.get(cont, np.nan),
            '90th Percentile': quant90_val_cont.get(cont, np.nan)
        })
        # All Ratio data
        continent_result.append({
            'Continent': cont,
            'Loss Indicator': col,
            'Stat Type': 'All Ratio data',
            'Mean Loss': all_mean_cont.get(cont, {}).get(col, np.nan),
            'Sample Count': all_count_cont.get(cont, {}).get(col, 0),
            '10th Percentile': quant10_val_cont.get(cont, np.nan),
            '90th Percentile': quant90_val_cont.get(cont, np.nan)
        })

# 5. Result Consolidation & Output
all_result = pd.DataFrame(world_result + continent_result)

# Pivot table for mean loss
result_combined = all_result.pivot_table(
    index=['Loss Indicator', 'Stat Type'],
    columns='Continent',
    values='Mean Loss',
    fill_value=np.nan
).round(6)

# Pivot table for sample count
count_combined = all_result.pivot_table(
    index=['Loss Indicator', 'Stat Type'],
    columns='Continent',
    values='Sample Count',
    fill_value=0
).astype(int)

# Extract percentile values (deduplicated)
quant_values = all_result[['Continent', '10th Percentile', '90th Percentile']].drop_duplicates().set_index('Continent')
quant_values = quant_values.round(6)

# Print final results
print("\n【0. Percentile Values of MeanRatio by Group】")
print(quant_values.to_string())

print("\n【1. Mean Loss Comparison (Low Ratio vs High Ratio vs All Data)】")
print(result_combined.to_string())

print("\n【2. Sample Count (Unit: rows)】")
print(count_combined.to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# -------------------------- Core Settings --------------------------
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (7, 3)
plt.rcParams['font.size'] = 15

# Continent-country ISO mapping
continent_iso_mapping = {
    'Africa': ['TZA', 'UGA', 'MDG', 'RWA'],
    'Asia': ['JOR', 'PAK', 'MYS', 'IDN'],
    'Europe': ['LVA', 'POL', 'BLR', 'UKR']
}

continents = ['Africa', 'Asia', 'Europe']
n_continents = len(continents)

# -------------------------- Data Extraction --------------------------
# Horizontal line data (Health loss 2050, scaled ×100)
health_loss_data = result_combined.loc['Health loss 2050']
stats_types = ['Ratio ≤ 10th percentile', 'All Ratio data']
plot_data = {}
for stat_type in stats_types:
    plot_data[stat_type] = [health_loss_data.loc[stat_type, cont] * 100 for cont in continents]

# Bar data (scaled ×100)
country_data = {}
for continent, isos in continent_iso_mapping.items():
    country_data[continent] = [df[df['ISO'] == iso]['Health loss 2050'].mean() * 100 for iso in isos]

# -------------------------- Precise Coordinate Calculation --------------------------
x_total_range = 3.0
continent_width = x_total_range / n_continents
continent_centers = [
    continent_width/2 + i*continent_width
    for i in range(n_continents)
]

n_bars_per_cont = 4
bar_width = 0.18
bar_gap = 0.05
total_bar_group_width = n_bars_per_cont * bar_width + (n_bars_per_cont-1)*bar_gap

# -------------------------- Plotting --------------------------
fig, ax = plt.subplots()

# ========== 1. Draw Bars ==========
bar_colors = {
    'Africa': ['#F2E8D8', '#D9C7AB', '#BFAA82', '#89704F'],
    'Asia': ['#C4D0D5', '#9CB0B8', '#73909B', '#4E6973'],
    'Europe': ['#D4D6E8', '#A9ACC9', '#7F82AA', '#414487']
}

country_xticks = []
country_labels = []

for i, cont in enumerate(continents):
    center = continent_centers[i]
    offsets = np.linspace(
        -total_bar_group_width/2 + bar_width/2,
         total_bar_group_width/2 - bar_width/2,
         n_bars_per_cont
    )
    bar_x = center + offsets

    ax.bar(
        x=bar_x,
        height=country_data[cont],
        width=bar_width,
        color=bar_colors[cont],
        alpha=0.8,
        zorder=2
    )

    country_xticks.extend(bar_x)
    country_labels.extend(continent_iso_mapping[cont])

# ========== 2. Draw Horizontal Lines ==========
line_gap = 0.03
alpha_value = 0.8

for i, cont in enumerate(continents):
    # ≤10th percentile: dashed
    ax.hlines(
        y=plot_data['Ratio ≤ 10th percentile'][i],
        xmin=i*continent_width + line_gap,
        xmax=(i+1)*continent_width - line_gap,
        color='#404040',
        linestyle='--',
        linewidth=1.5,
        alpha=alpha_value,
        zorder=4
    )

    # All data: solid
    ax.hlines(
        y=plot_data['All Ratio data'][i],
        xmin=i*continent_width + line_gap,
        xmax=(i+1)*continent_width - line_gap,
        color='#404040',
        linestyle='-',
        linewidth=1.5,
        alpha=alpha_value,
        zorder=4
    )

# ========== 3. Draw Continent Dividers ==========
for i in range(1, n_continents):
    ax.axvline(
        x=i*continent_width,
        color='#827F82', linestyle='--', linewidth=1.3, alpha=0.3, zorder=1
    )

# -------------------------- Axis Settings --------------------------
ax.set_xticks(continent_centers)
ax.set_xticklabels(continents, fontsize=15)

ax.set_xticks(country_xticks, minor=True)
ax.set_xticklabels(country_labels, minor=True, fontsize=13)

ax.tick_params(axis='x', which='major', pad=30)
ax.tick_params(axis='x', which='minor', pad=10)
ax.set_xlim(0, x_total_range)

ax.set_yticks([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
ax.set_yticklabels(['1.0', '2.0', '3.0', '4.0', '5.0', '6.0'])
ax.set_ylabel('Health loss 2050 (%)', fontsize=15)

all_y = (
    plot_data['Ratio ≤ 10th percentile'] +
    plot_data['All Ratio data'] +
    sum(country_data.values(), [])
)
ax.set_ylim(min(all_y) * 0.7, max(all_y) * 1.1)

# Frame
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_visible(True)

# -------------------------- Save & Show --------------------------
plt.tight_layout()
plt.savefig('C:\\...\\HealthLoss_Continent.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -------------------------- Core Settings --------------------------
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (3, 3)
plt.rcParams['font.size'] = 15

# -------------------------- Load and Prepare Data --------------------------
df = pd.read_excel(
    "E:\\...\\03Loss_Mainland_Combined_20251119.xlsx",
    sheet_name="SSP585"
)[lambda x: x['ID'].notna()]

TARGET_COL = 'Health loss 2050'
QUANTILE = 0.1
HIGH_QUANTILE = 0.9

all_data = df[TARGET_COL] * 100
q10_val = df['MeanRatio'].quantile(QUANTILE)
q10_data = df[df['MeanRatio'] <= q10_val][TARGET_COL] * 100
q90_val = df['MeanRatio'].quantile(HIGH_QUANTILE)
q90_data = df[df['MeanRatio'] >= q90_val][TARGET_COL] * 100

stats_data = {
    'all': {'mean': all_data.mean(), 'std': all_data.std(), 'raw': all_data},
    'q10': {'mean': q10_data.mean(), 'std': q10_data.std(), 'raw': q10_data},
    'q90': {'mean': q90_data.mean(), 'std': q90_data.std(), 'raw': q90_data}
}

# -------------------------- World Data --------------------------
continent_mapping = {
    'World': ['Top', 'Bot', 'Avg']
}
continents = ['World']

country_data = {
    'World': [
        stats_data['q90']['mean'],
        stats_data['q10']['mean'],
        stats_data['all']['mean']
    ]
}

std_data = {
    'World': [
        stats_data['q90']['std'],
        stats_data['q10']['std'],
        stats_data['all']['std']
    ]
}

raw_data = {
    'World': [
        stats_data['q90']['raw'],
        stats_data['q10']['raw'],
        stats_data['all']['raw']
    ]
}

# -------------------------- Coordinate Parameters --------------------------
x_total_range = 1.0
continent_centers = [0.5]

n_bars_per_cont = 3
bar_width = 0.20
bar_gap = 0.08

total_bar_group_width = (
    n_bars_per_cont * bar_width +
    (n_bars_per_cont - 1) * bar_gap
)

# -------------------------- Plotting --------------------------
fig, ax = plt.subplots()

bar_colors = {
    'World': ['#26A69A', '#00695C', '#004D40']
}

sub_xticks = []
sub_labels = []

for i, cont in enumerate(continents):
    center = continent_centers[i]

    offsets = np.linspace(
        -total_bar_group_width / 2 + bar_width / 2,
         total_bar_group_width / 2 - bar_width / 2,
         n_bars_per_cont
    )

    bar_x = center + offsets

    bars = ax.bar(
        bar_x,
        country_data[cont],
        width=bar_width,
        color=bar_colors[cont],
        alpha=0.85,
        edgecolor='black',
        linewidth=0.8
    )

    ax.errorbar(
        bar_x,
        country_data[cont],
        yerr=std_data[cont],
        fmt='none',
        ecolor='#404040',
        elinewidth=1.0,
        capsize=5,
        capthick=1.0,
        zorder=4
    )

    for j, x_pos in enumerate(bar_x):
        data_points = raw_data[cont][j]
        x_jitter = np.random.normal(0, bar_width/6, size=len(data_points))
        ax.scatter(
            x_pos + x_jitter,
            data_points,
            s=20,
            color=bar_colors[cont][j],
            edgecolor='black',
            linewidth=0.3,
            alpha=1,
            zorder=5
        )

    sub_xticks.extend(bar_x)
    sub_labels.extend(continent_mapping[cont])

# -------------------------- Axis Settings --------------------------
ax.set_xticks(sub_xticks)
ax.set_xticklabels(sub_labels, fontsize=15)
ax.tick_params(axis='x', which='major', pad=8)
ax.set_xlim(0, x_total_range)

group_center = np.mean(sub_xticks)

ax.text(
    group_center,
    -0.20,
    'World',
    ha='center',
    va='top',
    fontsize=15,
    transform=ax.get_xaxis_transform()
)

ax.set_ylim(-0.15, 6.0)

y_ticks = [0, 2.0, 4.0, 6.0]
ax.set_yticks(y_ticks)
ax.set_yticklabels([f'{tick:.1f}' if tick != 0 else f'{tick:.0f}' for tick in y_ticks], fontsize=15)

ax.set_ylabel('Health loss 2050 (%)', fontsize=15)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_visible(True)

# -------------------------- Save and Show --------------------------
plt.tight_layout()
plt.savefig(
    'C:\\...\\HealthLoss_World_dot.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# -------------------------- Core Settings --------------------------
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (7, 3)
plt.rcParams['font.size'] = 15

# Continent-country ISO mapping
continent_iso_mapping = {
    'Africa': ['TZA', 'ZMB', 'MDG', 'RWA'],
    'Asia': ['BGD', 'KHM', 'IDN', 'MYS'],
    'Europe': ['ESP', 'GRC', 'BLR', 'UKR']
}
continents = ['Africa', 'Asia', 'Europe']
n_continents = len(continents)

# -------------------------- Data Extraction --------------------------
# Horizontal line data (Total economic loss 2050, scaled ×100)
economic_loss_data = result_combined.loc['Total economic loss 2050']
stats_types = ['Ratio ≤ 10th percentile', 'All Ratio data']
plot_data = {}
for stat_type in stats_types:
    plot_data[stat_type] = [economic_loss_data.loc[stat_type, cont] * 100 for cont in continents]

# Bar data (scaled ×100)
country_data = {}
for continent, isos in continent_iso_mapping.items():
    country_data[continent] = [df[df['ISO'] == iso]['Total economic loss 2050'].mean() * 100 for iso in isos]

# -------------------------- Precise Coordinate Calculation --------------------------
x_total_range = 3.0
continent_width = x_total_range / n_continents
continent_centers = [
    continent_width/2 + i*continent_width
    for i in range(n_continents)
]

n_bars_per_cont = 4
bar_width = 0.18
bar_gap = 0.05
total_bar_group_width = n_bars_per_cont * bar_width + (n_bars_per_cont-1)*bar_gap

# -------------------------- Plotting --------------------------
fig, ax = plt.subplots()

# ========== 1. Draw Bars ==========
bar_colors = {
    'Africa': ['#F2E8D8', '#D9C7AB', '#BFAA82', '#89704F'],
    'Asia': ['#C4D0D5', '#9CB0B8', '#73909B', '#4E6973'],
    'Europe': ['#D4D6E8', '#A9ACC9', '#7F82AA', '#414487']
}

country_xticks = []
country_labels = []

for i, cont in enumerate(continents):
    center = continent_centers[i]
    offsets = np.linspace(
        -total_bar_group_width/2 + bar_width/2,
         total_bar_group_width/2 - bar_width/2,
         n_bars_per_cont
    )
    bar_x = center + offsets

    ax.bar(
        x=bar_x,
        height=country_data[cont],
        width=bar_width,
        color=bar_colors[cont],
        alpha=0.8,
        zorder=2
    )

    country_xticks.extend(bar_x)
    country_labels.extend(continent_iso_mapping[cont])

# ========== 2. Draw Horizontal Lines ==========
line_gap = 0.03
alpha_value = 0.8

for i, cont in enumerate(continents):
    # ≤10th percentile: dashed
    ax.hlines(
        y=plot_data['Ratio ≤ 10th percentile'][i],
        xmin=i*continent_width + line_gap,
        xmax=(i+1)*continent_width - line_gap,
        color='#404040',
        linestyle='--',
        linewidth=1.5,
        alpha=alpha_value,
        zorder=4
    )

    # All data: solid
    ax.hlines(
        y=plot_data['All Ratio data'][i],
        xmin=i*continent_width + line_gap,
        xmax=(i+1)*continent_width - line_gap,
        color='#404040',
        linestyle='-',
        linewidth=1.5,
        alpha=alpha_value,
        zorder=4
    )

# ========== 3. Draw Continent Dividers ==========
for i in range(1, n_continents):
    ax.axvline(
        x=i*continent_width,
        color='#827F82', linestyle='--', linewidth=1.3, alpha=0.3, zorder=1
    )

# -------------------------- Axis Settings --------------------------
ax.set_xticks(continent_centers)
ax.set_xticklabels(continents, fontsize=15)

ax.set_xticks(country_xticks, minor=True)
ax.set_xticklabels(country_labels, minor=True, fontsize=13)

ax.tick_params(axis='x', which='major', pad=30)
ax.tick_params(axis='x', which='minor', pad=10)
ax.set_xlim(0, x_total_range)

ax.set_yticks([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0])
ax.set_yticklabels(['1.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0'])
ax.set_ylabel('Total economic loss 2050 (%)', fontsize=15)

all_y = (
    plot_data['Ratio ≤ 10th percentile'] +
    plot_data['All Ratio data'] +
    sum(country_data.values(), [])
)
ax.set_ylim(min(all_y) * 0.7, max(all_y) * 1.1)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_visible(True)

# -------------------------- Save & Show --------------------------
plt.tight_layout()
plt.savefig('C:\\...\\TotalEconomicLoss_Continent.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -------------------------- Core Settings --------------------------
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (3, 3)
plt.rcParams['font.size'] = 15

# -------------------------- Load and Prepare Data --------------------------
df = pd.read_excel(
    "E:\\...\\03Loss_Mainland_Combined_20251119.xlsx",
    sheet_name="SSP585"
)[lambda x: x['ID'].notna()]

TARGET_COL = 'Total economic loss 2050'
QUANTILE = 0.1
HIGH_QUANTILE = 0.9

all_data = df[TARGET_COL] * 100
q10_val = df['MeanRatio'].quantile(QUANTILE)
q10_data = df[df['MeanRatio'] <= q10_val][TARGET_COL] * 100
q90_val = df['MeanRatio'].quantile(HIGH_QUANTILE)
q90_data = df[df['MeanRatio'] >= q90_val][TARGET_COL] * 100

stats_data = {
    'all': {'mean': all_data.mean(), 'std': all_data.std(), 'raw': all_data},
    'q10': {'mean': q10_data.mean(), 'std': q10_data.std(), 'raw': q10_data},
    'q90': {'mean': q90_data.mean(), 'std': q90_data.std(), 'raw': q90_data}
}

# -------------------------- World Data --------------------------
continent_mapping = {
    'World': ['Top', 'Bot', 'Avg']
}
continents = ['World']

country_data = {
    'World': [
        stats_data['q90']['mean'],
        stats_data['q10']['mean'],
        stats_data['all']['mean']
    ]
}

std_data = {
    'World': [
        stats_data['q90']['std'],
        stats_data['q10']['std'],
        stats_data['all']['std']
    ]
}

raw_data = {
    'World': [
        stats_data['q90']['raw'],
        stats_data['q10']['raw'],
        stats_data['all']['raw']
    ]
}

# -------------------------- Coordinate Parameters --------------------------
x_total_range = 1.0
continent_centers = [0.5]

n_bars_per_cont = 3
bar_width = 0.20
bar_gap = 0.08

total_bar_group_width = (
    n_bars_per_cont * bar_width +
    (n_bars_per_cont - 1) * bar_gap
)

# -------------------------- Plotting --------------------------
fig, ax = plt.subplots()

bar_colors = {
    'World': ['#26A69A', '#00695C', '#004D40']
}

sub_xticks = []
sub_labels = []

for i, cont in enumerate(continents):
    center = continent_centers[i]

    offsets = np.linspace(
        -total_bar_group_width / 2 + bar_width / 2,
         total_bar_group_width / 2 - bar_width / 2,
         n_bars_per_cont
    )

    bar_x = center + offsets

    bars = ax.bar(
        bar_x,
        country_data[cont],
        width=bar_width,
        color=bar_colors[cont],
        alpha=0.85,
        edgecolor='black',
        linewidth=0.8
    )

    ax.errorbar(
        bar_x,
        country_data[cont],
        yerr=std_data[cont],
        fmt='none',
        ecolor='#404040',
        elinewidth=1.0,
        capsize=5,
        capthick=1.0,
        zorder=4
    )

    for j, x_pos in enumerate(bar_x):
        data_points = raw_data[cont][j]
        x_jitter = np.random.normal(0, bar_width/6, size=len(data_points))
        ax.scatter(
            x_pos + x_jitter,
            data_points,
            s=20,
            color=bar_colors[cont][j],
            edgecolor='black',
            linewidth=0.3,
            alpha=1,
            zorder=5
        )

    sub_xticks.extend(bar_x)
    sub_labels.extend(continent_mapping[cont])

# -------------------------- Axis Settings --------------------------
ax.set_xticks(sub_xticks)
ax.set_xticklabels(sub_labels, fontsize=15)
ax.tick_params(axis='x', which='major', pad=8)
ax.set_xlim(0, x_total_range)

group_center = np.mean(sub_xticks)

ax.text(
    group_center,
    -0.20,
    'World',
    ha='center',
    va='top',
    fontsize=15,
    transform=ax.get_xaxis_transform()
)

ax.set_ylim(-0.15, 7.0)

y_ticks = [0, 2.0, 4.0, 6.0]
ax.set_yticks(y_ticks)
ax.set_yticklabels([f'{tick:.1f}' if tick != 0 else f'{tick:.0f}' for tick in y_ticks], fontsize=15)

ax.set_ylabel('Total economic loss 2050 (%)', fontsize=15)

for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_visible(True)

# -------------------------- Save and Show --------------------------
plt.tight_layout()
plt.savefig(
    'C:\\...\\Totaleconomicloss_World_dot.png',
    dpi=300,
    bbox_inches='tight'
)
plt.show()

In [ ]:
import pandas as pd

# 1. Data Loading and Basic Filtering
df = pd.read_excel(
    "E:\\...\\03Loss_Mainland_Combined_20251119.xlsx",
    sheet_name="SSP585"
)[lambda x: x['ID'].notna()]

# 2. Calculate 10th percentile of MeanRatio
ratio_10_percentile = df['MeanRatio'].quantile(0.1)
print(f"10th percentile of MeanRatio: {ratio_10_percentile:.6f}")

# 3. Core Configuration
cols = ['Total economic loss 2050', 'Economic loss 2050', 'Health loss 2050']

# Define 5 filter conditions
filters = {
    'NonLowRatio': df['MeanRatio'] > ratio_10_percentile,
    'World': df.index.notna(),
    'LowRatio': df['MeanRatio'] <= ratio_10_percentile,
    'LowRatio+NDGAIN75%': 
        (df['MeanRatio'] <= ratio_10_percentile) & (df['NDGAINRank'] == 'NDGAIN_Bottom75%'),
    'LowRatio+NDGAIN75%+LDC': 
        (df['MeanRatio'] <= ratio_10_percentile) & (df['NDGAINRank'] == 'NDGAIN_Bottom75%') & (df['LDC'] == 1)
}

# 4. Define calculation function
def calc_group_means():
    res_dict = {}
    for group_name, cond in filters.items():
        filtered_data = df[cond]
        mean_series = filtered_data[cols].mean().round(6)
        res_dict[f'{group_name} ({len(filtered_data)} rows)'] = mean_series
    return pd.DataFrame(res_dict)

# 5. Calculate and print results
df_results = calc_group_means()
print(f"\nLoss Mean Values by Group")
print(df_results.to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -------------------------- Core Settings --------------------------
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (3.4, 3)
plt.rcParams['font.size'] = 12

# -------------------------- Calculate Statistics --------------------------
TARGET_COL = 'Total economic loss 2050'
stats_data = {}

for group_name, cond in filters.items():
    filtered_data = df[cond][TARGET_COL] * 100
    stats_data[group_name] = {
        'mean': filtered_data.mean(),
        'std': filtered_data.std()
    }

# Organize data in specified order
group_order = ['NonLowRatio', 'World', 'LowRatio', 'LowRatio+NDGAIN75%', 'LowRatio+NDGAIN75%+LDC']
means = [stats_data[g]['mean'] for g in group_order]
stds = [stats_data[g]['std'] for g in group_order]

new_labels = [
    'Non-Low',
    'Global',
    'LowR',
    'LowR+\nNDGAIN',
    'LowR+\nNDGAIN+\nLDC'
]
labels = new_labels

# Reference values for horizontal lines
world_value = stats_data['World']['mean']
LowRatio_value = stats_data['LowRatio']['mean']

# -------------------------- Plotting --------------------------
n_bars = 5
bar_width = 0.5
step = 0.65
x_pos = np.arange(n_bars) * step
fig, ax = plt.subplots()

# Bar colors
bar_colors = [
    '#D1B38B',
    '#26685D',
    '#DE8452',
    '#CD5C5C',
    '#4C72B1'
]

# Draw bars and error bars
ax.bar(
    x_pos, means, bar_width,
    color=bar_colors, alpha=0.85,
    edgecolor='black', linewidth=0.8
)
ax.errorbar(
    x_pos, means, yerr=stds, fmt='none',
    ecolor='#404040', elinewidth=1.0,
    capsize=4, capthick=1.0, zorder=4
)

# -------------------------- Horizontal Reference Lines --------------------------
line_gap = -0.3
alpha_value = 0.8
xmin = x_pos[0] - (bar_width/10) + line_gap
xmax = x_pos[-1] + (bar_width/10) - line_gap

# Solid line: Global mean
ax.hlines(
    y=world_value, xmin=xmin, xmax=xmax,
    color='#404040', linestyle='-',
    linewidth=1.5, alpha=alpha_value, zorder=4
)

# Dashed line: LowRatio mean
ax.hlines(
    y=LowRatio_value, xmin=xmin, xmax=xmax,
    color='#404040', linestyle='--',
    linewidth=1.5, alpha=alpha_value, zorder=4
)

# -------------------------- Axis Settings --------------------------
ax.set_xticks(x_pos)
ax.set_xticklabels(labels, fontsize=10, ha='center')
ax.set_ylim(-0.10, 6.4)
ax.set_yticks([0, 2.0, 4.0, 6.0])
ax.set_yticklabels(['0', '2.0', '4.0', '6.0'], fontsize=12)
ax.set_ylabel('Total economic loss 2050 (%)', fontsize=12)

# Frame style
for spine in ax.spines.values():
    spine.set_linewidth(0.8)
    spine.set_visible(True)

# -------------------------- Save & Show --------------------------
plt.tight_layout()
plt.savefig(
    'C:\\...\\Totaleconomicloss_5groups.png',
    dpi=300, bbox_inches='tight'
)
plt.show()